# EDA — Predicting F1 Top-10 Finishes (2022–2024)

**Course:** IIT414W | **Group:** G19 | **Members:** Daniela Ávila, Klaus Krause

**Problem:** Predict whether a driver finishes in the Top 10 of a Formula 1 race.

**Data:** Race results 2022–2024 via the Jolpica API (`https://api.jolpi.ca/ergast/f1/`).

**Structure:** Every section follows `## Question → ### Data → ### Answer → ### Decision`.

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 414

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
import requests
import time
from scipy.stats import spearmanr, pointbiserialr, chi2_contingency

np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14,
})

SEASONS  = [2022, 2023, 2024]
BASE_URL = 'https://api.jolpi.ca/ergast/f1'
PALETTE  = ['#1A6FBF', '#D94F3D', '#2E9E5B', '#E8891A', '#7B4FBF', '#2AADAD']

print(f'RANDOM_SEED = {RANDOM_SEED}')

In [ ]:
# ── Data loading  (paginated — API default limit = 100) ──────────────────────
def fetch_season_results(year):
    """Fetch ALL race results for a season, handling Jolpica pagination."""
    url   = f'{BASE_URL}/{year}/results.json'
    rows, limit, offset = [], 100, 0
    while True:
        r = requests.get(url, params={'limit': limit, 'offset': offset}, timeout=30)
        r.raise_for_status()
        data  = r.json()['MRData']
        total = int(data['total'])
        for race in data['RaceTable']['Races']:
            for res in race['Results']:
                rows.append({
                    'season':           int(race['season']),
                    'round':            int(race['round']),
                    'race_name':        race['raceName'],
                    'circuit_id':       race['Circuit']['circuitId'],
                    'circuit_name':     race['Circuit']['circuitName'],
                    'country':          race['Circuit']['Location']['country'],
                    # PRE-RACE features
                    'driver_id':        res['Driver']['driverId'],
                    'driver_name':      res['Driver']['givenName'] + ' ' + res['Driver']['familyName'],
                    'constructor_id':   res['Constructor']['constructorId'],
                    'constructor_name': res['Constructor']['name'],
                    'grid':             int(res['grid']),
                    # POST-RACE outcomes (reference only — NOT features)
                    'position_text':    res['positionText'],
                    'status':           res['status'],
                    'laps':             int(res['laps']),
                    'points':           float(res['points']),
                })
        offset += limit
        if offset >= total:
            break
        time.sleep(0.3)
    return rows

all_rows = []
for yr in SEASONS:
    print(f'Fetching {yr}...')
    all_rows.extend(fetch_season_results(yr))
    time.sleep(0.5)

df_raw = pd.DataFrame(all_rows)
print(f'\n{len(df_raw):,} driver-race entries | seasons: {sorted(df_raw["season"].unique())}')
print('Rounds per season:', df_raw.groupby('season')['round'].nunique().to_dict())

In [ ]:
# ── Target variable & derived columns ────────────────────────────────────────
df_raw['position_num']   = pd.to_numeric(df_raw['position_text'], errors='coerce')
df_raw['top_10']         = (df_raw['position_num'] <= 10).astype(int)
df_raw['finished']       = df_raw['position_num'].notna().astype(int)
df_raw['pit_lane_start'] = (df_raw['grid'] == 0).astype(int)

df = df_raw.copy()
print('Dataset shape:', df.shape)
print('\nTarget distribution (%):')
print(df['top_10'].value_counts(normalize=True)
      .rename({0: 'non-top-10', 1: 'top-10'})
      .mul(100).round(1))
df.head(3)

---

# Section 1 — Data Quality Audit

> Requirement 3.7: missing values (MCAR/MAR/MNAR classification for ≥3 columns), data types, outliers, temporal availability.

In [ ]:
# ── Missing values & data types ──────────────────────────────────────────────
print('=== DTYPES ===')
print(df.dtypes)

print('\n=== NULL COUNTS ===')
null_counts = df.isnull().sum()
print('No nulls.' if null_counts.sum() == 0 else null_counts[null_counts > 0])

print('\n=== Non-classified finishes (position_num NaN) ===')
print(df['position_num'].isna().sum(), 'rows')
print(df[df['position_num'].isna()]['status'].value_counts().head(10))

print('\n=== grid distribution ===')
print(df['grid'].describe().round(2))
print(f'Pit-lane starters (grid=0): {df["pit_lane_start"].sum()}')

### Data Quality Findings

| Column | Issue | Classification | Decision |
|---|---|---|---|
| `position_num` | Non-numeric codes (R, D, W…) for DNF/DSQ | **MNAR** — retirement is caused by race outcomes, not random | Keep; `top_10 = 0` for all non-classified entries |
| `grid` | Value = 0 for pit-lane starters | **MAR** — real event, not recording error | Flag via `pit_lane_start`; exclude from grid analyses |
| `top_10` | Derived from `position_num`; no true NaNs | N/A | Clean |
| `points` | Post-race outcome — leakage risk | N/A | Excluded from all features |
| `laps` | Post-race outcome — leakage risk | N/A | Excluded from all features |
| `status` | Post-race categorical (Finished / Retired…) | N/A | Reference only |

> **Temporal availability:** `grid`, `driver_id`, `constructor_id`, `circuit_id`, `season`, `round` → known **pre-race** (safe as features). `position_num`, `points`, `laps`, `status` → known **post-race** (leakage if used as features).

> Full log → `DATA_QUALITY_LOG.md`.

---

# Section 2 — Research Questions

> Requirement 3.1: ≥5 questions, each with Question → Plot → Interpretation → Decision.

## Q1 — Is the dataset balanced? (Class Balance Analysis)

**Question:** What is the distribution of Top-10 vs. non-Top-10 finishes? Is accuracy a safe evaluation metric?

In [ ]:
### Data
counts      = df['top_10'].map({1: 'Top-10', 0: 'Non-Top-10'}).value_counts()
percentages = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)

# ── Bar chart ──
ax = axes[0]
bars = ax.bar(counts.index, counts.values,
              color=[PALETTE[0], PALETTE[1]], width=0.45,
              edgecolor='white', linewidth=2)
ax.set_ylim(0, counts.max() * 1.28)
ax.set_title('Entry Count by Finish Category\n2022–2024', fontsize=13, pad=10)
ax.set_ylabel('Driver-race entries')
ax.set_xlabel('')
for bar, (lbl, pct) in zip(bars, percentages.items()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2,
            h + counts.max() * 0.03,
            f'{int(h):,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Pie chart ──
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=[PALETTE[0], PALETTE[1]], startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    textprops={'fontsize': 11},
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[1].set_title('Class Balance', fontsize=13, pad=10)

fig.suptitle('Class Balance Analysis — Target Variable', fontsize=14, y=1.02)
plt.show()

print(f'Always-predict-Top-10 accuracy:     {percentages["Top-10"]:.1f}%')
print(f'Always-predict-Non-Top-10 accuracy: {percentages["Non-Top-10"]:.1f}%')

### Answer

The dataset is **nearly perfectly balanced** (~50% Top-10, ~50% non-Top-10). This is structurally expected — exactly 10 of ~20 drivers per race finish in the top 10. A naive always-predict-Top-10 baseline achieves ~50% accuracy — the theoretical floor.

### Decision

Accuracy is a **reasonable metric** for this balanced dataset. Any heuristic scoring ≤ 50% adds no value. The 50% floor is evaluated formally in `baseline.ipynb`.

## Q2 — Is the Top-10 rate stable across the 2022, 2023, and 2024 seasons? (Temporal Pattern Analysis)

**Question:** Does the class distribution or feature distribution shift between seasons? This determines whether a temporal split produces a representative validation set.

In [ ]:
### Data
season_stats = df.groupby('season').agg(
    total=('top_10', 'count'),
    top10=('top_10', 'sum'),
    top10_rate=('top_10', 'mean'),
    races=('round', 'nunique'),
    drivers=('driver_id', 'nunique'),
).reset_index()
season_stats['top10_pct'] = season_stats['top10_rate'] * 100

grid_mean = df[df['pit_lane_start'] == 0].groupby('season')['grid'].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
s_labels  = season_stats['season'].astype(str).tolist()
s_colors  = [PALETTE[2], PALETTE[0], PALETTE[3]]

def annotate_bars(ax, values, fmt='{:.1f}', pad_frac=0.04):
    ymax = max(values)
    ax.set_ylim(0, ymax * 1.28)
    for i, v in enumerate(values):
        ax.text(i, v + ymax * pad_frac, fmt.format(v),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Top-10 rate
axes[0].bar(s_labels, season_stats['top10_pct'], color=s_colors,
            edgecolor='white', linewidth=2, width=0.5)
axes[0].axhline(50, color='#D94F3D', linestyle='--', linewidth=1.2,
                alpha=0.7, label='50% floor')
axes[0].set_title('Top-10 Rate per Season', pad=10)
axes[0].set_ylabel('Top-10 finish rate (%)')
axes[0].set_ylim(42, 58)
axes[0].legend(loc='lower right')
for i, v in enumerate(season_stats['top10_pct']):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')

# Mean grid
gvals = grid_mean.values
axes[1].bar(grid_mean.index.astype(str), gvals,
            color=s_colors, edgecolor='white', linewidth=2, width=0.5)
axes[1].set_title('Mean Starting Grid Position', pad=10)
axes[1].set_ylabel('Mean grid position')
annotate_bars(axes[1], gvals, fmt='{:.1f}')

# Total entries
tvals = season_stats['total'].values
axes[2].bar(s_labels, tvals, color=s_colors, edgecolor='white', linewidth=2, width=0.5)
axes[2].set_title('Total Driver-Race Entries', pad=10)
axes[2].set_ylabel('Entries')
ax2 = axes[2]
ax2.set_ylim(0, max(tvals) * 1.28)
for i, (v, row) in enumerate(zip(tvals, season_stats.itertuples())):
    ax2.text(i, v + max(tvals) * 0.04,
             f'{v:,}\n({row.races} races)',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

fig.suptitle('Temporal Pattern Analysis — 2022 vs 2023 vs 2024', fontsize=14, y=1.03)
plt.show()
print(season_stats[['season','races','total','top10_pct','drivers']].to_string(index=False))

### Answer

The Top-10 rate is **stable across all three seasons** (~50% each year). Mean grid position is consistent (~10.5), indicating no systematic shift in qualifying distributions. 2024 has slightly more races due to calendar expansion.

### Decision

A **temporal split is appropriate and safe** — no distribution shift that would make a 2022–2023-trained heuristic invalid for 2024.

## Q3 — Does grid position predict a Top-10 finish?

**Question:** Is there a monotonic relationship between starting grid position and probability of finishing in the Top 10? This is the foundation of our baseline heuristic.

In [ ]:
### Data
df_grid   = df[df['pit_lane_start'] == 0].copy()
grid_top10 = df_grid.groupby('grid').agg(
    top10_rate=('top_10', 'mean'),
    count=('top_10', 'count')
).reset_index()
grid_top10['top10_pct'] = grid_top10['top10_rate'] * 100

rho, p_val = spearmanr(df_grid['grid'], df_grid['top_10'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

# ── Bar chart by grid position ──
ax = axes[0]
bar_c = [PALETTE[0] if g <= 10 else PALETTE[1] for g in grid_top10['grid']]
ax.bar(grid_top10['grid'], grid_top10['top10_pct'],
       color=bar_c, edgecolor='white', linewidth=0.6, width=0.85)
ax.axvline(10.5, color='#333', linestyle='--', linewidth=1.2, alpha=0.6)
ax.axhline(50,   color='#D94F3D', linestyle=':', linewidth=1.2, alpha=0.7)
ax.set_xlim(0.4, 20.6)
ax.set_ylim(0, 105)
ax.set_title(f'Top-10 Rate by Grid Position\nSpearman \u03c1\u202f=\u202f{rho:.3f}  (p\u202f<\u202f0.001)',
             pad=10)
ax.set_xlabel('Grid position  (1 = pole)')
ax.set_ylabel('Top-10 finish rate (%)')
ax.set_xticks(range(1, 21))
ax.legend(handles=[
    Patch(fc=PALETTE[0], label='Grid 1\u201310'),
    Patch(fc=PALETTE[1], label='Grid 11\u201320'),
    plt.Line2D([0],[0], color='#333',    linestyle='--', label='P10/P11 boundary'),
    plt.Line2D([0],[0], color='#D94F3D', linestyle=':',  label='50\u202f% floor'),
], loc='upper right')

# ── Smoothed trend ──
ax2 = axes[1]
rolling = (grid_top10.set_index('grid')['top10_pct']
           .rolling(window=3, center=True, min_periods=1).mean())
ax2.plot(rolling.index, rolling.values, 'o-',
         color=PALETTE[0], linewidth=2.2, markersize=6, label='Smoothed rate (3-race window)')
ax2.fill_between(grid_top10['grid'],
                 grid_top10['top10_pct'] - 6,
                 grid_top10['top10_pct'] + 6,
                 alpha=0.12, color=PALETTE[0], label='\u00b16 pp uncertainty band')
ax2.axvline(10.5, color='#333',    linestyle='--', linewidth=1.2, alpha=0.6)
ax2.axhline(50,   color='#D94F3D', linestyle=':',  linewidth=1.2, alpha=0.7)
ax2.set_xlim(0.4, 20.6)
ax2.set_ylim(0, 105)
ax2.set_title('Smoothed Trend: Top-10 Rate vs Grid', pad=10)
ax2.set_xlabel('Grid position')
ax2.set_ylabel('Top-10 finish rate (%)')
ax2.set_xticks(range(1, 21))
ax2.legend(loc='upper right')

fig.suptitle('Grid Position as Predictor of Top-10 Finish', fontsize=14, y=1.03)
plt.show()
print(f'Spearman \u03c1 = {rho:.4f}   p = {p_val:.2e}')
print(f'Top-10 rate  grid\u202f1\u201310:  {df_grid[df_grid["grid"]<=10]["top_10"].mean()*100:.1f}%')
print(f'Top-10 rate  grid 11\u201320: {df_grid[df_grid["grid"]>10]["top_10"].mean()*100:.1f}%')

### Answer

Grid position has a **strong negative Spearman correlation** with Top-10 probability (ρ ≈ −0.50, p < 0.001). Drivers starting 1–10 finish top-10 ~70–80% of the time; starting 11–20 only ~20–30%. The relationship is monotonic but not deterministic — positions 9–12 show high variance.

### Decision

**Grid position is the single strongest pre-race predictor.** The rule "grid ≤ 10 → predict top-10" is the natural domain heuristic, evaluated in `baseline.ipynb`.

## Q4 — Which constructors are most likely to place drivers in the Top 10?

**Question:** Do certain teams systematically outperform their grid position? Does constructor identity add independent signal?

In [ ]:
### Data
constructor_stats = (
    df.groupby('constructor_name')
    .agg(top10_rate=('top_10','mean'), entries=('top_10','count'), mean_grid=('grid','mean'))
    .reset_index()
    .sort_values('top10_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)

cmap = plt.cm.RdYlGn
norm = plt.Normalize(0, 1)
bar_colors = [cmap(norm(v)) for v in constructor_stats['top10_rate']]

bars = ax.barh(constructor_stats['constructor_name'],
               constructor_stats['top10_rate'] * 100,
               color=bar_colors, edgecolor='white', linewidth=1.2, height=0.65)
ax.axvline(50, color='#D94F3D', linestyle='--', linewidth=1.2, alpha=0.7, label='50% reference')

max_val = constructor_stats['top10_rate'].max() * 100
ax.set_xlim(0, max_val * 1.45)

for bar, (_, row) in zip(bars, constructor_stats.iterrows()):
    xpos = bar.get_width() + max_val * 0.02
    ypos = bar.get_y() + bar.get_height() / 2
    ax.text(xpos, ypos,
            f"{row['top10_rate']*100:.0f}%  ·  {row['entries']} starts  ·  mean grid {row['mean_grid']:.1f}",
            va='center', fontsize=8.5, color='#333')

ax.set_title('Top-10 Finish Rate by Constructor (2022\u20132024)', pad=12)
ax.set_xlabel('Top-10 finish rate (%)')
ax.set_ylabel('')
ax.legend(loc='lower right')
plt.show()

### Answer

Constructor performance is **highly stratified**: top teams achieve top-10 rates well above 50%; backmarkers stay consistently below. The gap spans ~40–60 pp and confirms that team-specific factors add **independent signal** beyond grid position alone.

### Decision

**Constructor identity is the most valuable additional feature** for Lab 2. The grid-based heuristic implicitly captures some of this, but explicitly encoding constructor improves predictions for mid-field uncertainty.

## Q5 — Are individual drivers consistently Top-10 performers across seasons?

**Question:** Do some drivers systematically over- or under-perform their grid position year-over-year? Would driver history add signal beyond constructor?

In [ ]:
### Data
driver_stats = (
    df.groupby('driver_name')
    .agg(top10_rate=('top_10','mean'), entries=('top_10','count'), mean_grid=('grid','mean'))
    .reset_index()
)
# Keep drivers with >= 2 full-season equivalent starts (robust to partial calendars)
min_starts = max(15, int(df.groupby('season')['round'].nunique().mean() * 0.6))
driver_stats = driver_stats[driver_stats['entries'] >= min_starts].sort_values('top10_rate', ascending=True)

n_drivers  = len(driver_stats)
fig_height = max(6, n_drivers * 0.38)
fig, ax    = plt.subplots(figsize=(11, fig_height), constrained_layout=True)

cmap_d   = plt.cm.RdYlGn
bar_cd   = [cmap_d(norm(v)) for v in driver_stats['top10_rate']]
bars_d   = ax.barh(driver_stats['driver_name'],
                   driver_stats['top10_rate'] * 100,
                   color=bar_cd, edgecolor='white', linewidth=1, height=0.7)
ax.axvline(50, color='#D94F3D', linestyle='--', linewidth=1.2, alpha=0.7, label='50% reference')
max_dr = driver_stats['top10_rate'].max() * 100
ax.set_xlim(0, max_dr * 1.3)

for bar, (_, row) in zip(bars_d, driver_stats.iterrows()):
    ax.text(bar.get_width() + max_dr * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{row['top10_rate']*100:.0f}%",
            va='center', fontsize=9, color='#333')

ax.set_title(f'Top-10 Finish Rate by Driver (\u2265{min_starts} starts, 2022\u20132024)', pad=12)
ax.set_xlabel('Top-10 finish rate (%)')
ax.set_ylabel('')
ax.legend(loc='lower right')
plt.show()

# Year-on-year stability
d_szn = df.groupby(['driver_name','season'])['top_10'].mean().unstack()
consistency = d_szn.std(axis=1).dropna().sort_values()
print('Most consistent (low season-to-season std):')
print(consistency.head(5).round(3))
print('\nLeast consistent:')
print(consistency.tail(5).round(3))

### Answer

Driver top-10 rates range ~10–90% and are **stable across seasons**. Elite drivers maintain high rates year-over-year; backmarkers stay consistently low. However, driver performance is **highly correlated with constructor** — the best drivers drive for the best teams.

### Decision

**Driver identity adds signal but is partly redundant with constructor.** Test marginal gain in Lab 2. For now, grid-based heuristic is sufficient.

## Q6 — Has constructor dominance shifted across 2022–2024?

**Question:** Did the competitive ordering between teams change? Distribution shifts in constructor performance matter for whether training on 2022–2023 generalises to 2024.

In [ ]:
### Data
top_c = df.groupby('constructor_name')['top_10'].count().nlargest(8).index.tolist()

const_pivot = (
    df[df['constructor_name'].isin(top_c)]
    .groupby(['constructor_name','season'])['top_10'].mean()
    .unstack().fillna(0)
    .sort_values(by=2024, ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

# ── Heatmap ──
sns.heatmap(
    const_pivot * 100,
    annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=0.5, linecolor='white',
    ax=axes[0], cbar_kws={'label': 'Top-10 rate (%)', 'shrink': 0.8},
    annot_kws={'fontsize': 11, 'fontweight': 'bold'}
)
axes[0].set_title('Constructor Top-10 Rate by Season (%)', pad=10)
axes[0].set_xlabel('Season')
axes[0].set_ylabel('')
axes[0].tick_params(axis='y', rotation=0)

# ── Line plot ──
line_colors = plt.cm.tab10(np.linspace(0, 0.9, len(const_pivot)))
for (constructor, row), color in zip(const_pivot.iterrows(), line_colors):
    axes[1].plot(const_pivot.columns, row * 100,
                 marker='o', linewidth=2, markersize=7,
                 label=constructor, color=color)
axes[1].axhline(50, color='#D94F3D', linestyle=':', linewidth=1.2,
                alpha=0.7, label='50% ref')
axes[1].set_title('Constructor Top-10 Rate Trend (2022\u20132024)', pad=10)
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Top-10 finish rate (%)')
axes[1].set_xticks([2022, 2023, 2024])
axes[1].set_ylim(-5, 110)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8.5, framealpha=0.9)

fig.suptitle('Constructor Dominance Shift Across Seasons', fontsize=14, y=1.03)
plt.show()

### Answer

Constructor dominance is **partially stable** but shows meaningful shifts: Red Bull peaked in 2023, Mercedes declined, McLaren rose toward 2024. This confirms a moderate distribution shift between training (2022–2023) and test (2024) data.

### Decision

The temporal shift reinforces the need for a **temporal split**. For Lab 2, constructor features should use recent-season rates, not all-time averages.

---

# Section 3 — Correlation Analysis

> Requirement 3.4: Correlation for ≥5 candidate features. Using Spearman (ordinal), Cramér's V (categorical), point-biserial (binary). Interpreting magnitude AND direction.

In [ ]:
### Data
def cramers_v(x, y):
    ct   = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n    = ct.to_numpy().sum()
    r, k = ct.shape
    return float(np.sqrt(chi2 / n / (min(r, k) - 1))) if min(r, k) > 1 else 0.0

df_corr = df[df['pit_lane_start'] == 0].copy()

rho_grid,   p_grid   = spearmanr(df_corr['grid'],   df_corr['top_10'])
rho_season, p_season = spearmanr(df_corr['season'], df_corr['top_10'])
cv_constr  = cramers_v(df_corr['constructor_id'], df_corr['top_10'])
cv_driver  = cramers_v(df_corr['driver_id'],      df_corr['top_10'])
cv_circuit = cramers_v(df_corr['circuit_id'],     df_corr['top_10'])
rpb_pit, p_pit = pointbiserialr(df['pit_lane_start'], df['top_10'])

corr_data = [
    ('grid',           'Numeric',     'Spearman \u03c1',    rho_grid,   'Negative: higher grid \u2192 fewer top-10'),
    ('season',         'Numeric',     'Spearman \u03c1',    rho_season, '\u22480: no year trend'),
    ('constructor_id', 'Categorical', 'Cram\u00e9r\u2019s V',   cv_constr,  'Team performance'),
    ('driver_id',      'Categorical', 'Cram\u00e9r\u2019s V',   cv_driver,  'Correlated with constructor'),
    ('circuit_id',     'Categorical', 'Cram\u00e9r\u2019s V',   cv_circuit, 'Weak: limited signal over grid'),
    ('pit_lane_start', 'Binary',      'Point-biserial', rpb_pit,   'Negative: pit-lane \u2192 rarely top-10'),
]
corr_table = pd.DataFrame(corr_data, columns=['Feature','Type','Measure','Value','Notes'])
corr_table['Value'] = corr_table['Value'].round(4)
print(corr_table.to_string(index=False))

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
vals_abs = corr_table['Value'].abs()
orig_vals = corr_table['Value']
bar_fill = [PALETTE[1] if v < 0 else PALETTE[0] for v in orig_vals]

bars = ax.bar(corr_table['Feature'], vals_abs, color=bar_fill,
              edgecolor='white', linewidth=1.5, width=0.55)
ax.set_ylim(0, vals_abs.max() * 1.35)
ax.set_title("Feature\u2013Target Association Strength\n(|Spearman \u03c1| or Cram\u00e9r's V)", pad=10)
ax.set_xlabel('Feature')
ax.set_ylabel('|Association|')
ax.legend(handles=[
    Patch(fc=PALETTE[0], label='Positive / no direction (Cram\u00e9r)'),
    Patch(fc=PALETTE[1], label='Negative direction'),
], loc='upper right')

for bar, orig, val in zip(bars, orig_vals, vals_abs):
    sign = '\u2212' if orig < 0 else ''
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + vals_abs.max() * 0.03,
            f'{sign}{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.show()

### Correlation Analysis Summary

| Feature | Measure | Value | Direction | Interpretation |
|---|---|---|---|---|
| `grid` | Spearman ρ | ~−0.50 | **Negative** | Strong: lower grid → more top-10s. Primary predictor. |
| `season` | Spearman ρ | ~0.00 | Negligible | No year-over-year trend. |
| `constructor_id` | Cramér's V | ~0.40 | N/A | Moderate–strong: team matters. |
| `driver_id` | Cramér's V | ~0.50 | N/A | Strong but partly redundant with constructor. |
| `circuit_id` | Cramér's V | ~0.10–0.15 | N/A | Weak: limited signal over grid. |
| `pit_lane_start` | Point-biserial | ~−0.10 | **Negative** | Pit-lane starters rarely top-10. |

---

# Section 4 — Trap Awareness: Survivorship Bias in Driver Historical Rates

> Requirement 3.5: Explicitly check for one of the three traps.

**Trap:** *Survivorship bias* — drivers who underperform lose their seats. Our dataset over-represents drivers good enough to keep racing, biasing their observed top-10 rates upward.

In [ ]:
### Data
drv_nseas = df.groupby('driver_id')['season'].nunique().rename('seasons_present')
drv_rate  = df.groupby('driver_id')['top_10'].mean().rename('top10_rate')
drv_name  = df.groupby('driver_id')['driver_name'].first()
drv_df    = pd.concat([drv_nseas, drv_rate, drv_name], axis=1).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# ── Bar: seasons present distribution ──
sc = drv_df['seasons_present'].value_counts().sort_index()
ax = axes[0]
bars0 = ax.bar(sc.index.astype(str), sc.values,
               color=[PALETTE[1], PALETTE[3], PALETTE[2]],
               edgecolor='white', linewidth=2, width=0.5)
ax.set_ylim(0, sc.max() * 1.3)
ax.set_title('Seasons Per Driver in Dataset\n(survivorship indicator)', pad=10)
ax.set_xlabel('Seasons present (out of 3)')
ax.set_ylabel('Number of drivers')
for bar, v in zip(bars0, sc.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + sc.max() * 0.04,
            str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')

# ── Scatter + mean: top-10 rate vs seasons ──
ax2  = axes[1]
jitter = np.random.default_rng(RANDOM_SEED)
for n_s, grp in drv_df.groupby('seasons_present'):
    jx = jitter.uniform(-0.12, 0.12, len(grp))
    ax2.scatter(n_s + jx, grp['top10_rate'] * 100,
                alpha=0.55, s=55, color=PALETTE[n_s - 1], label=f'{n_s} season(s)')

means_s = drv_df.groupby('seasons_present')['top10_rate'].mean()
ax2.plot(means_s.index, means_s.values * 100, 'k--o',
         linewidth=2, markersize=9, zorder=5, label='Group mean')

for x, y in zip(means_s.index, means_s.values * 100):
    ax2.text(x + 0.08, y + 1.5, f'{y:.1f}%',
             fontsize=9, fontweight='bold', color='black')

ax2.set_title('Top-10 Rate vs. Seasons Present\n(survivorship bias check)', pad=10)
ax2.set_xlabel('Seasons driver appears in dataset')
ax2.set_ylabel('Top-10 finish rate (%)')
ax2.set_xticks([1, 2, 3])
ax2.legend(loc='upper left')

fig.suptitle('Survivorship Bias \u2014 Driver Selection Effect', fontsize=14, y=1.03)
plt.show()

print('Mean top-10 rate by seasons present:')
print((drv_df.groupby('seasons_present')['top10_rate'].mean() * 100).round(1))

### What We Found

Drivers present in **all 3 seasons** have systematically higher top-10 rates than one-season drivers — consistent with survivorship bias. Low performers lose their seats; multi-season drivers are a self-selected high-performance group.

**Trap confirmed.** Using raw driver historical top-10 rates as features without survivorship correction would overstate driver-level predictive power and fail for rookies (cold-start problem). Recommended fix for Lab 2: recency-weighted rates + rookie flag.

---

# Section 5 — Temporal Train / Validation / Test Split

> Requirement 3.6: Explicit temporal split with written rationale. No random splits.

| Set | Data | Purpose |
|---|---|---|
| **Train** | 2022 + 2023 | Learn historical patterns |
| **Validation** | 2024 rounds 1–(N/2) | Evaluate baseline — **only set used in Lab 1** |
| **Test** | 2024 rounds (N/2+1)–end | **Held out entirely — NOT TOUCHED until Lab 2** |

**Why temporal?** F1 data is time-ordered. A random split would leak 2024 performance into training, inflating accuracy in a way that doesn't generalise to real prediction scenarios.

In [ ]:
### Define split
total_2024 = int(df[df['season'] == 2024]['round'].max())
val_end    = total_2024 // 2
test_start = val_end + 1

print(f'Total 2024 rounds: {total_2024}')
print(f'Validation : 2024 rounds 1\u2013{val_end}')
print(f'Test       : 2024 rounds {test_start}\u2013{total_2024}  \u2190 HELD OUT')

train_mask = df['season'].isin([2022, 2023])
val_mask   = (df['season'] == 2024) & (df['round'] <= val_end)
test_mask  = (df['season'] == 2024) & (df['round'] >= test_start)

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print('\n=== SPLIT SUMMARY ===')
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{name:6s}: {len(split):5,} rows | {split["round"].nunique():2d} races '
          f'| top-10 rate: {split["top_10"].mean()*100:.1f}%')

# ── Leakage checks ───────────────────────────────────────────────────────────
assert df_train['season'].max() <= 2023,    'LEAKAGE: 2024 in train!'
assert df_val['season'].eq(2024).all(),     'Val must be 2024 only'
assert df_val['round'].max() <= val_end,    'LEAKAGE: test rounds in val!'
assert df_test['round'].min() >= test_start,'LEAKAGE: val rounds in test!'
assert not set(df_val.index) & set(df_test.index), 'Overlapping val/test!'
print('\n\u2713 All leakage checks passed.')

In [ ]:
# ── Visualise split ───────────────────────────────────────────────────────────
df['_split'] = 'Unknown'
df.loc[train_mask, '_split'] = 'Train (2022\u20132023)'
df.loc[val_mask,   '_split'] = 'Validation (2024 H1)'
df.loc[test_mask,  '_split'] = 'Test (2024 H2) \u2014 Held out'

split_colors = {
    'Train (2022\u20132023)':       PALETTE[2],
    'Validation (2024 H1)':        PALETTE[0],
    'Test (2024 H2) \u2014 Held out': PALETTE[1],
}

df['_x'] = df['season'] + (df['round'] - 1) / 26

fig, ax = plt.subplots(figsize=(13, 3.2), constrained_layout=True)
for label, color in split_colors.items():
    sub = df[df['_split'] == label]
    ax.scatter(sub['_x'], [label] * len(sub),
               color=color, s=22, alpha=0.45, rasterized=True)

ax.set_title('Temporal Train / Validation / Test Split', pad=10)
ax.set_xlabel('Season (approximate position along calendar)')
ax.set_yticks(list(split_colors.keys()))
ax.set_xlim(2021.8, 2025.1)
ax.axvline(2024, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.text(2024.02, 0.02, '2024 start', transform=ax.get_xaxis_transform(),
        fontsize=8, color='gray', va='bottom')
plt.show()

df.drop(columns=['_split', '_x'], inplace=True)

### Split Rationale

- **Train (2022–2023):** Two full seasons — sufficient history, covers two regulatory eras.
- **Validation (2024 H1):** Used exclusively to evaluate the baseline in `baseline.ipynb`. Reflects current-season car performance.
- **Test (2024 H2):** Completely untouched. Never used in any analysis, feature selection, or threshold tuning. Reserved for Lab 2.

All three splits maintain ~50% top-10 rate, confirming no label leakage.

---

# Section 6 — 1-3-1 Summary

> Requirement 3.8: Final Markdown cell with 1-3-1 summary — same text submitted on Canvas.

---

## Grid position is the dominant pre-race signal — build the baseline around it, and add constructor in Lab 2.

**Three evidence points:**

1. **Grid position has the strongest pre-race correlation with Top-10 finish** (Spearman ρ ≈ −0.50, p < 0.001): drivers starting 1–10 finish in the top 10 ~70–80% of the time, while those starting 11–20 do so only ~20–30%. This single feature makes a rule-based heuristic viable.

2. **Constructor identity is the strongest additional signal** (Cramér's V ≈ 0.40): top teams achieve top-10 rates ~70–100%, while backmarkers stay below 30% even from mid-field starts. The gap spans ~50 pp and is stable across seasons (with moderate shifts confirming the need for a temporal split).

3. **Survivorship bias inflates driver historical top-10 rates**: drivers present across all 3 seasons show ~10–15 pp higher top-10 rates than one-season drivers. Using raw driver rates without bias correction overstates driver-level predictive power and fails for rookies.

**Action:** Implement "grid ≤ 10 → predict Top-10" in `baseline.ipynb`, evaluated on the 2024 H1 validation set. Keep 2024 H2 as an untouched test set. In Lab 2, add constructor encoding and apply survivorship-bias correction to driver features.